#PERSIAPAN LIBRARY

In [1]:
!pip install ultralytics #menginstal Ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.1/903.1 kB 23.9 MB/s eta 0:00:00


In [2]:
import os  # Operasi file dan direktori
import random  # Generator angka acak
import shutil  # Manajemen file dan folder
import zipfile  # Kompresi dan ekstraksi file ZIP
import csv  # Membaca/menulis file CSV
import matplotlib.pyplot as plt  # Visualisasi data
import seaborn as sns  # Visualisasi statistik
from PIL import Image  # Pengolahan gambar
import pandas as pd  # Analisis data berbasis tabel
import numpy as np  # Operasi matematika dan array
from ultralytics import YOLO  # Deteksi objek AI
from google.colab import drive  # untuk mengakses Google Drive

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


#Mount Google Drive

In [4]:
# Mengatur direktori untuk dataset
drive.mount('/content/gdrive')  # # Menghubungkan Google Drive ke Colab
HOME = os.getcwd()  # Mendapatkan direktori kerja saat ini
print("Current Directory:", HOME)  # Menampilkan direktori kerja saat ini
!mkdir {HOME}/data  # Membuat folder baru bernama 'data' di direktori saat ini
%cd {HOME}/data  # # Berpindah ke folder 'data' yang baru dibuat

Mounted at /content/gdrive


In [9]:
# Mendownload file ZIP dan mengekstraknya
zip_file = "/content/gdrive/MyDrive/belajarmachineL/minor-data-slayer.zip"  # Path ke file ZIP di Google Drive
with zipfile.ZipFile(zip_file, 'r') as zip_ref:  # Membuka file ZIP untuk dibaca
    zip_ref.extractall(HOME)  # Mengekstrak isi file ZIP ke direktori kerja

In [10]:
# Tentukan konstanta
HOME = '/content'
ZIP_FILE = "/content/gdrive/MyDrive/belajarmachineL/minor-data-slayer.zip"
SOURCE_DIR = os.path.join(HOME, 'train') # Direktori sumber data training
DEST_DIR = os.path.join(HOME, 'data') # Direktori tujuan untuk data
TEST_DIR = os.path.join(HOME, 'test') # Direktori data pengujian
CLASS_NAMES = [ # Daftar label kelas untuk dataset
    "backward_falls", "forward_falls", "jumping", "laying", "left_falls",
    "picking", "right_falls", "sitting_falls", "squat",
    "standing_falls", "stretching", "walking"
]

In [11]:
# Fungsi untuk mengekstrak file zip
def extract_zip(zip_file, extract_path): # Mendefinisikan fungsi untuk ekstraksi file ZIP
    with zipfile.ZipFile(zip_file, 'r') as zip_ref: # Membuka file ZIP untuk dibaca
        zip_ref.extractall(extract_path) # Mengekstrak isi file ZIP ke direktori kerja

# Data Processing

In [12]:
# Fungsi untuk membuat direktori
def create_directories(base_path, class_names):  # Mendefinisikan fungsi untuk membuat direktori
    for category in ['train', 'validation']:  # Iterasi untuk kategori 'train' dan 'validation'
        category_path = os.path.join(base_path, category)  # Membuat path kategori
        os.makedirs(category_path, exist_ok=True)  # Membuat folder kategori jika belum ada
        for class_name in class_names:  # Iterasi untuk setiap nama kelas
            os.makedirs(os.path.join(category_path, class_name), exist_ok=True)  # Membuat folder kelas

In [13]:
# Fungsi untuk membagi data menjadi data latih dan validasi
def split_data(source_dir, dest_dir, class_names, split_ratio=(0.8, 0.2), seed=42):  # Membagi data ke train dan validation
    random.seed(seed)  # Mengatur seed untuk hasil acak yang konsisten
    for class_name in class_names:  # Iterasi setiap nama kelas
        class_dir = os.path.join(source_dir, class_name)  # Path direktori untuk kelas tertentu
        if not os.path.exists(class_dir):  # Periksa apakah direktori kelas ada
            print(f"Warning: Directory {class_dir} does not exist. Skipping.")  # Peringatan jika direktori tidak ada
            continue

        images = os.listdir(class_dir)  # Daftar file gambar di direktori kelas
        if not images:  # Periksa apakah ada gambar dalam direktori
            print(f"Warning: No images found in {class_dir}. Skipping.")  # Peringatan jika tidak ada gambar
            continue

        random.shuffle(images)  # Mengacak urutan gambar
        split_index = int(split_ratio[0] * len(images))  # Hitung indeks pembagian data
        train_images = images[:split_index]  # Data gambar untuk training
        val_images = images[split_index:]  # Data gambar untuk validation

        for img, subset in zip([train_images, val_images], ['train', 'validation']):  # Iterasi untuk subset data
            for image in img:  # Iterasi setiap gambar dalam subset
                src = os.path.join(class_dir, image)  # Path sumber gambar
                dst = os.path.join(dest_dir, subset, class_name, image)  # Path tujuan gambar
                try:
                    shutil.copy(src, dst)  # Menyalin gambar dari sumber ke tujuan
                except Exception as e:  # Tangani kesalahan jika terjadi selama penyalinan
                    print(f"Error copying {src} to {dst}: {e}")  # Pesan kesalahan

In [14]:
# Fungsi untuk menghitung item dalam direktori
def count_items_in_directory(directory):  # Mendefinisikan fungsi untuk menghitung item dalam direktori
    return sum(len(files) for _, _, files in os.walk(directory))  # Menghitung total file dalam direktori menggunakan os.walk

In [15]:
# Fungsi untuk menampilkan statistik dataset
def count_items(base_path, class_names):  # Mendefinisikan fungsi untuk menampilkan statistik dataset
    for category in ['train', 'validation']:  # Iterasi untuk kategori 'train' dan 'validation'
        print(f"--- {category.capitalize()} Data ---")  # Menampilkan header kategori
        for class_name in class_names:  # Iterasi untuk setiap nama kelas
            class_path = os.path.join(base_path, category, class_name)  # Path untuk direktori kelas
            num_items = count_items_in_directory(class_path) if os.path.exists(class_path) else 0  # Hitung jumlah gambar dalam kelas, jika ada
            print(f"{class_name.capitalize()}: {num_items} images")  # Menampilkan jumlah gambar per kelas

In [16]:
# Fungsi untuk memeriksa gambar dalam direktori
def check_images_in_directory(directory):  # Mendefinisikan fungsi untuk memeriksa validitas gambar dalam direktori
    for root, _, files in os.walk(directory):  # Iterasi melalui semua subdirektori dan file dalam direktori
        for file in files:  # Iterasi setiap file dalam direktori
            img_path = os.path.join(root, file)  # Membuat path lengkap untuk setiap file gambar
            try:
                img = Image.open(img_path)  # Membuka gambar untuk memverifikasi
                img.verify()  # Memverifikasi apakah gambar valid
            except Exception as e:  # Menangani kesalahan jika gambar tidak valid
                print(f"Invalid file found: {img_path}, Error: {e}")  # Menampilkan file gambar yang tidak valid beserta pesan kesalahan

In [17]:
# Mengekstrak dan mengatur data
extract_zip(ZIP_FILE, HOME)  # Mengekstrak file ZIP ke direktori utama
create_directories(DEST_DIR, CLASS_NAMES)  # Membuat direktori 'train' dan 'validation' untuk setiap kelas
split_data(SOURCE_DIR, DEST_DIR, CLASS_NAMES)  # Membagi data gambar ke dalam subset 'train' dan 'validation' berdasarkan kelas

In [18]:
# Menampilkan statistik kumpulan data
count_items(DEST_DIR, CLASS_NAMES)  # Menampilkan statistik jumlah gambar dalam kategori 'train' dan 'validation'
check_images_in_directory(os.path.join(DEST_DIR, 'train'))  # Memeriksa validitas gambar dalam direktori 'train'
check_images_in_directory(os.path.join(DEST_DIR, 'validation'))  # Memeriksa validitas gambar dalam direktori 'validation'

--- Train Data ---
Backward_falls: 231 images
Forward_falls: 233 images
Jumping: 431 images
Laying: 416 images
Left_falls: 225 images
Picking: 545 images
Right_falls: 249 images
Sitting_falls: 320 images
Squat: 421 images
Standing_falls: 288 images
Stretching: 487 images
Walking: 448 images
--- Validation Data ---
Backward_falls: 47 images
Forward_falls: 47 images
Jumping: 87 images
Laying: 84 images
Left_falls: 45 images
Picking: 109 images
Right_falls: 50 images
Sitting_falls: 64 images
Squat: 85 images
Standing_falls: 58 images
Stretching: 98 images
Walking: 90 images


#Modeling

In [19]:
# Melatih YOLO model
os.environ['WANDB_MODE'] = 'disabled'  # Menonaktifkan integrasi dengan Weights & Biases untuk pelatihan model
model = YOLO('yolo11m-cls.pt')  # Memuat model YOLO yang telah dilatih sebelumnya (yolo11m-cls.pt)
model.train(data=DEST_DIR, epochs=20, imgsz=256, batch=32, patience=5)  # Melatih model YOLO dengan data, jumlah epoch, ukuran gambar, batch size, dan patience

100%|██████████| 22.4M/22.4M [00:00<00:00, 42.4MB/s]


Ultralytics 8.3.54 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
engine/trainer: task=classify, mode=train, model=yolo11m-cls.pt, data=/content/data, epochs=20, time=None, patience=5, batch=32, imgsz=256, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, li

100%|██████████| 5.35M/5.35M [00:00<00:00, 284MB/s]


AMP: checks passed ✅


train: Scanning /content/data/train... 4294 images, 0 corrupt: 100%|██████████| 4294/4294 [00:01<00:00, 3952.52it/s]

...

                   all          1          1



20 epochs completed in 0.494 hours.
Optimizer stripped from runs/classify/train/weights/last.pt, 20.9MB
Optimizer stripped from runs/classify/train/weights/best.pt, 20.9MB

Validating runs/classify/train/weights/best.pt...
Ultralytics 8.3.54 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
YOLO11m-cls summary (fused): 138 layers, 10,357,004 parameters, 0 gradients, 39.3 GFLOPs
train: /content/data/train... found 4294 images in 12 classes ✅ 
val: /content/data/validation... found 864 images in 12 classes ✅ 
test: /content/data/test... found 2152 images in 1 classes: ERROR ❌️ requires 12 classes, not 1


               classes   top1_acc   top5_acc: 100%|██████████| 14/14 [00:11<00:00,  1.18it/s]


                   all          1          1
Speed: 0.1ms preprocess, 1.0ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to runs/classify/train


ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f70ce6fadd0>
curves: []
curves_results: []
fitness: 1.0
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 1.0, 'metrics/accuracy_top5': 1.0, 'fitness': 1.0}
save_dir: PosixPath('runs/classify/train')
speed: {'preprocess': 0.11511560943391587, 'inference': 1.0385590570944327, 'loss': 0.00024669700198703346, 'postprocess': 0.00045117404725816514}
task: 'classify'
top1: 1.0
top5: 1.0